# handlers

> Structure decomposition workflow handlers — init, split, merge, undo, reset, AI split

In [ ]:
#| default_exp routes.decomposition.handlers

In [ ]:
#| export
from typing import List, Dict, Any, Tuple

from cjm_fasthtml_card_stack.core.models import CardStackState
from cjm_fasthtml_card_stack.core.constants import DEFAULT_VISIBLE_COUNT, DEFAULT_CARD_WIDTH

from cjm_fasthtml_interactions.core.state_store import get_session_id

from cjm_fasthtml_workflow_transcript_decomp.core.models import WorkingSegment
from cjm_fasthtml_workflow_transcript_decomp.routes.models import DecompUrls
from cjm_fasthtml_workflow_transcript_decomp.components.step_decomposition.step_renderer import (
    _render_decomposition_content, render_decomp_stats, render_toolbar
)
from cjm_fasthtml_workflow_transcript_decomp.components.step_decomposition.card_stack_config import (
    DECOMP_TS_IDS,
)
from cjm_fasthtml_workflow_transcript_decomp.services.text_utils import (
    word_index_to_char_position,
)
from cjm_fasthtml_workflow_transcript_decomp.services.segmentation import (
    split_segment_at_position, merge_segments, reindex_segments,
    reconstruct_source_blocks
)
from cjm_fasthtml_workflow_transcript_decomp.services.history import pop_history
from cjm_fasthtml_workflow_transcript_decomp.routes.decomposition.core import (
    _to_segments, _load_decomp_context, _get_decomp_state,
    _get_selection_state, _update_decomp_state, _push_history,
    _build_card_stack_state,
)
from cjm_fasthtml_workflow_transcript_decomp.routes.decomposition.card_stack import (
    _build_slots_oob, _build_nav_response
)

from cjm_fasthtml_workflow_transcript_decomp.workflow.workflow import StructureDecompWorkflow

## Mutation Response Builder

Assembles the full OOB response for handlers that mutate segment data.
Includes decomposition-specific elements (stats, toolbar) in addition to card stack elements.

In [ ]:
#| export
def _build_mutation_response(
    segment_dicts: List[Dict[str, Any]],  # Serialized segments
    focused_index: int,  # Currently focused segment index
    visible_count: int,  # Number of visible cards
    history_depth: int,  # Current undo history depth
    urls: DecompUrls,  # URL bundle
    is_split_mode: bool = False,  # Whether split mode is active
) -> Tuple:  # OOB elements (slots + progress + focus + stats + toolbar)
    """Build the standard OOB response for mutation handlers."""
    state = CardStackState(
        focused_index=focused_index,
        visible_count=visible_count,
        active_mode="split" if is_split_mode else None,
    )

    # Library handles: slots + progress + focus
    nav_response = _build_nav_response(segment_dicts, state, urls)

    # Decomposition-specific OOB elements
    segments = _to_segments(segment_dicts)
    stats_oob = render_decomp_stats(segments, oob=True)
    toolbar_oob = render_toolbar(
        reset_url=urls.reset, ai_split_url=urls.ai_split, undo_url=urls.undo,
        can_undo=(history_depth > 0), visible_count=visible_count, oob=True,
    )
    return (*nav_response, stats_oob, toolbar_oob)

## Initialize Handler

In [ ]:
#| export
async def _handle_decomp_init(
    workflow: StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess,  # FastHTML session object
    urls: DecompUrls,  # URL bundle for decomposition routes
    visible_count: int = DEFAULT_VISIBLE_COUNT,  # Number of visible cards
    card_width: int = DEFAULT_CARD_WIDTH,  # Card stack width in rem
):  # Full decomposition step content with keyboard navigation
    """Initialize segments from Phase 1 selected sources."""
    session_id = get_session_id(sess)
    # Get selected sources from Phase 1
    selection_state = _get_selection_state(workflow, session_id)
    selected_sources = selection_state.get("selected_sources", [])
    
    # Read stored viewport preferences (may exist from previous session)
    decomp_state = _get_decomp_state(workflow, session_id)
    stored_visible_count = decomp_state.get("visible_count", visible_count)
    stored_card_width = decomp_state.get("card_width", card_width)
    
    if not selected_sources:
        # No sources selected, initialize with empty state
        _update_decomp_state(
            workflow, session_id,
            segments=[], initial_segments=[],
            is_initialized=True, focused_index=0,
            history=[], visible_count=stored_visible_count,
            card_width=stored_card_width,
        )
        return _render_decomposition_content(
            segments=[], focused_index=0, history_depth=0,
            visible_count=stored_visible_count, card_width=stored_card_width,
            urls=urls,
        )
    
    # Fetch source blocks via service API
    source_blocks = workflow.source_service.get_source_blocks(selected_sources)
    
    # Use segmentation service to split into sentences
    working_segments = await workflow.segmentation_service.split_combined_sources_async(
        source_blocks
    )
    segment_dicts = [s.to_dict() for s in working_segments]
    
    # Store in state
    _update_decomp_state(
        workflow, session_id,
        segments=segment_dicts,
        initial_segments=segment_dicts.copy(),
        is_initialized=True, focused_index=0,
        history=[], visible_count=stored_visible_count,
        card_width=stored_card_width,
    )
    
    return _render_decomposition_content(
        segments=_to_segments(segment_dicts),
        focused_index=0, history_depth=0,
        visible_count=stored_visible_count, card_width=stored_card_width,
        urls=urls,
    )

## Split Handler

In [ ]:
#| export
async def _handle_decomp_split(
    workflow:StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess,  # FastHTML session object
    segment_index:int,  # Index of segment to split
    urls:DecompUrls,  # URL bundle for decomposition routes
):  # OOB slot updates with stats, progress, focus, and toolbar
    """Split a segment at the specified word position."""
    session_id = get_session_id(sess)
    ctx = _load_decomp_context(workflow, session_id)

    # Extract word index from token selector hidden input
    form = await request.form()
    word_index = int(form.get(DECOMP_TS_IDS.anchor_name, 0))

    # Validate index
    if segment_index < 0 or segment_index >= len(ctx.segment_dicts):
        state = _build_card_stack_state(ctx)
        return _build_slots_oob(ctx.segment_dicts, state, urls)

    # Push current state to history before modification
    history_depth = _push_history(workflow, session_id, ctx.segment_dicts, segment_index)

    # Get the segment and convert word index to character position
    segment = WorkingSegment.from_dict(ctx.segment_dicts[segment_index])
    char_position = word_index_to_char_position(segment.text, word_index)

    # Can't split at beginning or end
    if char_position <= 0 or char_position >= len(segment.text):
        return _build_mutation_response(
            ctx.segment_dicts, segment_index, ctx.visible_count, history_depth, urls,
        )

    # Split the segment
    first_seg, second_seg = split_segment_at_position(segment, char_position)

    # Build and reindex new segments list
    new_segments = ctx.segment_dicts[:segment_index]
    new_segments.append(first_seg.to_dict())
    new_segments.append(second_seg.to_dict())
    new_segments.extend(ctx.segment_dicts[segment_index + 1:])

    reindexed = reindex_segments(_to_segments(new_segments))
    new_segment_dicts = [s.to_dict() for s in reindexed]

    # Update state — focus moves to the new segment (second half)
    new_focused_index = segment_index + 1
    _update_decomp_state(workflow, session_id,
        segments=new_segment_dicts, focused_index=new_focused_index,
    )

    return _build_mutation_response(
        new_segment_dicts, new_focused_index, ctx.visible_count, history_depth, urls,
    )

## Merge Handler

In [ ]:
#| export
def _handle_decomp_merge(
    workflow: StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess,  # FastHTML session object
    segment_index: int,  # Index of segment to merge (merges with previous)
    urls: DecompUrls,  # URL bundle for decomposition routes
):  # OOB slot updates with stats, progress, focus, and toolbar
    """Merge a segment with the previous segment."""
    session_id = get_session_id(sess)
    ctx = _load_decomp_context(workflow, session_id)
    
    # Can't merge first segment (nothing before it)
    if segment_index <= 0 or segment_index >= len(ctx.segment_dicts):
        state = _build_card_stack_state(ctx)
        return _build_slots_oob(ctx.segment_dicts, state, urls)
    
    # Push current state to history
    history_depth = _push_history(workflow, session_id, ctx.segment_dicts, segment_index)
    
    # Merge segments
    prev_segment = WorkingSegment.from_dict(ctx.segment_dicts[segment_index - 1])
    curr_segment = WorkingSegment.from_dict(ctx.segment_dicts[segment_index])
    merged = merge_segments(prev_segment, curr_segment)
    
    # Build and reindex new segments list
    new_segments = ctx.segment_dicts[:segment_index - 1]
    new_segments.append(merged.to_dict())
    new_segments.extend(ctx.segment_dicts[segment_index + 1:])
    
    reindexed = reindex_segments(_to_segments(new_segments))
    new_segment_dicts = [s.to_dict() for s in reindexed]
    
    # Update state — focus moves to merged segment (previous position)
    new_focused_index = segment_index - 1
    _update_decomp_state(workflow, session_id,
        segments=new_segment_dicts, focused_index=new_focused_index,
    )
    
    return _build_mutation_response(
        new_segment_dicts, new_focused_index, ctx.visible_count, history_depth, urls,
    )

## Undo Handler

In [ ]:
#| export
def _handle_decomp_undo(
    workflow: StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess,  # FastHTML session object
    urls: DecompUrls,  # URL bundle for decomposition routes
):  # OOB slot updates with stats, progress, focus, and toolbar
    """Undo the last operation by restoring previous state from history."""
    session_id = get_session_id(sess)
    ctx = _load_decomp_context(workflow, session_id)
    
    result = pop_history(ctx.history)
    if result is None:
        state = _build_card_stack_state(ctx)
        return _build_slots_oob(ctx.segment_dicts, state, urls)
    
    previous_segments, new_focused_index, remaining_history = result
    
    _update_decomp_state(workflow, session_id,
        segments=previous_segments, history=remaining_history,
        focused_index=new_focused_index,
    )
    
    return _build_mutation_response(
        previous_segments, new_focused_index, ctx.visible_count, len(remaining_history), urls,
    )

## Reset Handler

In [ ]:
#| export
def _handle_decomp_reset(
    workflow: StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess,  # FastHTML session object
    urls: DecompUrls,  # URL bundle for decomposition routes
):  # OOB slot updates with stats, progress, focus, and toolbar
    """Reset segments to the initial NLTK split result."""
    session_id = get_session_id(sess)
    ctx = _load_decomp_context(workflow, session_id)
    decomp_state = _get_decomp_state(workflow, session_id)
    initial_segments = decomp_state.get("initial_segments", [])
    
    # Push current state to history before reset
    history_depth = 0
    if ctx.segment_dicts:
        history_depth = _push_history(workflow, session_id, ctx.segment_dicts, ctx.focused_index)
    
    # Restore initial segments — reset focus to first segment
    _update_decomp_state(workflow, session_id,
        segments=initial_segments.copy(), focused_index=0,
    )
    
    return _build_mutation_response(
        initial_segments, 0, ctx.visible_count, history_depth, urls,
    )

## AI Split Handler

In [ ]:
#| export
async def _handle_decomp_ai_split(
    workflow: StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess,  # FastHTML session object
    urls: DecompUrls,  # URL bundle for decomposition routes
):  # OOB slot updates with stats, progress, focus, and toolbar
    """Re-run AI (NLTK) sentence splitting on all current text."""
    session_id = get_session_id(sess)
    ctx = _load_decomp_context(workflow, session_id)
    
    if not ctx.segment_dicts:
        state = _build_card_stack_state(ctx)
        return _build_slots_oob([], state, urls)
    
    # Push current state to history
    history_depth = _push_history(workflow, session_id, ctx.segment_dicts, ctx.focused_index)
    
    # Reconstruct source blocks from current segments
    source_blocks = reconstruct_source_blocks(ctx.segment_dicts)
    
    # Re-run NLTK splitting
    working_segments = await workflow.segmentation_service.split_combined_sources_async(
        source_blocks
    )
    new_segment_dicts = [s.to_dict() for s in working_segments]
    
    # Update state — reset focus to first segment
    _update_decomp_state(workflow, session_id,
        segments=new_segment_dicts, focused_index=0,
    )
    
    return _build_mutation_response(
        new_segment_dicts, 0, ctx.visible_count, history_depth, urls,
    )

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()